In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
khesng_uav_forest_sunny_next_path = kagglehub.dataset_download('khesng/uav-forest-sunny-next')
leehien12_uav_forest_sunny_part1_path = kagglehub.dataset_download('leehien12/uav-forest-sunny-part1')

print('Data source import complete.')


# Forest Segmentation - Training

In [ ]:
# ============================================================
# CONFIG - thay doi o day
# ============================================================
MODEL_NAME  = 'deeplabv3plus'  # unet | unetpp | deeplabv3plus
ENCODER     = 'mit_b2'        # resnet34 | resnet50 | mit_b2 | tu-convnext_tiny
IMG_SIZE    = (512, 512)
BATCH_SIZE  = 4
EPOCHS      = 50
LR          = 1e-3
NUM_WORKERS = 2
PATIENCE    = 3
SAVE_TOP_K  = 3
USE_AMP     = True

In [ ]:
# ============================================================
# SETUP
# ============================================================
!pip install -q segmentation-models-pytorch albumentations

import os, time, json, re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

IS_KAGGLE = os.path.exists('/kaggle')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUTPUT_DIR = Path('/kaggle/working/outputs') if IS_KAGGLE else Path('outputs')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Device: {DEVICE} | Kaggle: {IS_KAGGLE}')
if DEVICE == 'cuda': print(f'GPU: {torch.cuda.get_device_name()}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:0000:01
Device: cuda | Kaggle: True
GPU: Tesla T4


In [ ]:
# ============================================================
# GOOGLE DRIVE AUTO-BACKUP (optional)
# ============================================================
_gdrive = None
_folder_id = None

if IS_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        from google.oauth2.service_account import Credentials
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaFileUpload
        s = UserSecretsClient()
        _gdrive = build('drive', 'v3', credentials=Credentials.from_service_account_info(
            json.loads(s.get_secret('GDRIVE_CREDENTIALS')),
            scopes=['https://www.googleapis.com/auth/drive.file']))
        _folder_id = s.get_secret('GDRIVE_FOLDER_ID')
        print(f'[OK] Google Drive ready')
    except Exception as e:
        print(f'[SKIP] Drive: {e}')

def upload_to_drive(path):
    if not _gdrive: return
    try:
        f = Path(path)
        old = _gdrive.files().list(q=f"name='{f.name}' and '{_folder_id}' in parents and trashed=false",
                                   fields='files(id)').execute().get('files', [])
        media = MediaFileUpload(str(f), mimetype='application/octet-stream', resumable=True)
        if old: _gdrive.files().update(fileId=old[0]['id'], media_body=media).execute()
        else:   _gdrive.files().create(body={'name': f.name, 'parents': [_folder_id]}, media_body=media).execute()
        print(f'  [DRIVE] {f.name} ({f.stat().st_size/1024**2:.1f}MB)')
    except Exception as e:
        print(f'  [DRIVE FAIL] {e}')

[OK] Google Drive ready


In [ ]:
# ============================================================
# DATA
# ============================================================
LABEL_COLORS = [(0,255,255),(0,127,0),(19,132,69),(0,53,65),(130,76,0),
                (152,251,152),(151,126,171),(250,150,0),(115,176,195),(123,123,123),(0,0,0)]
CLASS_NAMES = ['Sky','Deciduous_trees','Coniferous_trees','Fallen_trees','Dirt_ground',
               'Ground_vegetation','Rocks','Building','Fence','Car','Empty']
NUM_CLASSES = len(CLASS_NAMES)

def rgb_to_mask(rgb):
    out = np.full(rgb.shape[:2], NUM_CLASSES-1, dtype=np.int64)
    for i, c in enumerate(LABEL_COLORS): out[np.all(rgb == c, axis=-1)] = i
    return out

# --- Tim data: dung os.walk (follow symlinks) ---
if IS_KAGGLE:
    DATA_ROOT = Path('/kaggle/working/data')
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for dirpath, dirnames, filenames in os.walk('/kaggle/input', followlinks=True):
        if os.path.basename(dirpath) == 'color':
            seq_dir = os.path.dirname(dirpath)  # parent of color/
            seq_name = os.path.basename(seq_dir)
            if re.match(r'seq\d+', seq_name):
                link = DATA_ROOT / seq_name
                if not link.exists():
                    os.symlink(seq_dir, link)
                    print(f'  Found: {seq_name} -> {seq_dir}')
else:
    DATA_ROOT = Path('data/forest_sunny')

seqs = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir() and d.name.startswith('seq')])
print(f'\nDATA: {DATA_ROOT}')
print(f'Sequences ({len(seqs)}): {seqs}')
if len(seqs) == 0:
    raise RuntimeError('Khong tim thay sequence nao! Kiem tra Add Data tren Kaggle.')

# --- Dataset ---
class ForestDataset(Dataset):
    def __init__(self, root, sequences, transform=None):
        self.transform = transform
        self.samples = []
        root = Path(root)
        for seq in sequences:
            cd = root / seq / 'color'
            if not cd.exists(): print(f'  [SKIP] {seq}'); continue
            for f in sorted(cd.glob('*.png')):
                lf = root / seq / 'labels' / f.name
                if lf.exists(): self.samples.append((f, lf))
        print(f'  Dataset: {len(self.samples)} samples from {sequences}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        img = np.array(Image.open(self.samples[i][0]).convert('RGB'))
        mask = rgb_to_mask(np.array(Image.open(self.samples[i][1]).convert('RGB')))
        if self.transform:
            t = self.transform(image=img, mask=mask); img, mask = t['image'], t['mask']
        return img.float(), mask.long()

# --- Split ---
if len(seqs) >= 4: split = {'train': seqs[:-2], 'val': [seqs[-2]], 'test': [seqs[-1]]}
elif len(seqs) == 3: split = {'train': seqs[:1], 'val': [seqs[1]], 'test': [seqs[2]]}
else: split = {'train': seqs, 'val': seqs[-1:], 'test': seqs[-1:]}
for k, v in split.items(): print(f'  {k}: {v}')

train_tf = A.Compose([A.Resize(*IMG_SIZE), A.HorizontalFlip(p=0.5), A.Rotate(limit=15, p=0.3, border_mode=0),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.3), A.HueSaturationValue(10,20,20,p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])
val_tf = A.Compose([A.Resize(*IMG_SIZE), A.Normalize(mean=(0.485,0.456,0.406), std=(0.229,0.224,0.225)), ToTensorV2()])

train_ds = ForestDataset(DATA_ROOT, split['train'], train_tf)
val_ds   = ForestDataset(DATA_ROOT, split['val'], val_tf)
if len(train_ds) == 0:
    raise RuntimeError(f'Train dataset empty! Sequences: {split["train"]}')

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
print(f'\n  Train: {len(train_ds)} | Val: {len(val_ds)}')

  Found: seq6 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq6/seq6
  Found: seq7 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq7/seq7
  Found: seq8 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq8/seq8
  Found: seq5 -> /kaggle/input/datasets/khesng/uav-forest-sunny-next/seq5/seq5
  Found: seq2 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq2/seq2
  Found: seq4 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq4/seq4
  Found: seq3 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq3/seq3
  Found: seq1 -> /kaggle/input/datasets/leehien12/uav-forest-sunny-part1/seq1/seq1

DATA: /kaggle/working/data
Sequences (8): ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq6', 'seq7', 'seq8']
  train: ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq6']
  val: ['seq7']
  test: ['seq8']
  Dataset: 8754 samples from ['seq1', 'seq2', 'seq3', 'seq4', 'seq5', 'seq6']
  Dataset: 1436 samples from ['seq7']

  Train: 8754 | Val: 1436


In [ ]:
# ============================================================
# MODEL
# ============================================================
MODELS = {
    'unet': lambda: smp.Unet(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'unetpp': lambda: smp.UnetPlusPlus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
    'deeplabv3plus': lambda: smp.DeepLabV3Plus(encoder_name=ENCODER, encoder_weights='imagenet', in_channels=3, classes=NUM_CLASSES),
}
model = MODELS[MODEL_NAME]().to(DEVICE)
print(f'{MODEL_NAME} ({ENCODER}): {sum(p.numel() for p in model.parameters()):,} params')

class CEDiceLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss()
    def forward(self, logits, targets):
        ce = self.ce(logits, targets)
        probs = F.softmax(logits, dim=1)
        tgt = F.one_hot(targets, NUM_CLASSES).permute(0,3,1,2).float()
        inter = (probs * tgt).sum(dim=(0,2,3))
        card  = probs.sum(dim=(0,2,3)) + tgt.sum(dim=(0,2,3))
        dice  = 1 - ((2*inter + 1e-6) / (card + 1e-6)).mean()
        return ce + 0.5 * dice

criterion = CEDiceLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = GradScaler('cuda', enabled=USE_AMP)

config.json:   0%|          | 0.00/135 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/98.9M [00:00<?, ?B/s]

deeplabv3plus (mit_b2): 25,351,643 params


In [ ]:
# ============================================================
# TRAIN
# ============================================================
best_miou = 0.0
no_improve = 0
saved = []

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # --- Train ---
    model.train()
    train_loss, n = 0.0, 0
    for imgs, masks in tqdm(train_loader, desc=f'E{epoch} train'):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        with autocast('cuda', enabled=USE_AMP):
            loss = criterion(model(imgs), masks)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); optimizer.zero_grad()
        train_loss += loss.item(); n += 1
    train_loss /= n

    # --- Val ---
    model.eval()
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    val_loss, n = 0.0, 0
    with torch.no_grad():
        for imgs, masks in tqdm(val_loader, desc=f'E{epoch} val'):
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            with autocast('cuda', enabled=USE_AMP):
                out = model(imgs)
                val_loss += criterion(out, masks).item(); n += 1
            p = out.argmax(1).cpu().numpy().flatten()
            t = masks.cpu().numpy().flatten()
            v = (t >= 0) & (t < NUM_CLASSES)
            cm += np.bincount(t[v]*NUM_CLASSES + p[v], minlength=NUM_CLASSES**2).reshape(NUM_CLASSES, NUM_CLASSES)
    val_loss /= n
    inter = np.diag(cm); union = cm.sum(1) + cm.sum(0) - inter
    iou = np.where(union > 0, inter/union, 0.0)
    miou = iou[union > 0].mean()
    scheduler.step()

    print(f'E{epoch}/{EPOCHS} | loss {train_loss:.4f}/{val_loss:.4f} | mIoU {miou:.4f} | {time.time()-t0:.0f}s')

    # --- Checkpoint ---
    if miou > best_miou:
        best_miou = miou; no_improve = 0
        path = CKPT_DIR / f'epoch_{epoch:03d}_miou_{miou:.4f}.pth'
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(), 'metric': miou}, path)
        saved.append((miou, path))
        saved.sort(key=lambda x: x[0])
        while len(saved) > SAVE_TOP_K:
            _, old = saved.pop(0)
            if old.exists(): old.unlink()
        upload_to_drive(str(path))
        print(f'  * NEW BEST {miou:.4f}')
    else:
        no_improve += 1
    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch}'); break

print(f'\nDone! Best mIoU: {best_miou:.4f}')

E1 val: 100%|██████████| 359/359 [05:26<00:00,  1.10it/s]
/tmp/ipykernel_55/522868119.py:39: RuntimeWarning: invalid value encountered in divide
  iou = np.where(union > 0, inter/union, 0.0)


E1/50 | loss 0.8120/0.5967 | mIoU 0.3937 | 2513s
  [DRIVE FAIL] <HttpError 403 when requesting None returned "Service Accounts do not have storage quota. Leverage shared drives (https://developers.google.com/workspace/drive/api/guides/about-shareddrives), or use OAuth delegation (http://support.google.com/a/answer/7281227) instead.". Details: "[{'message': 'Service Accounts do not have storage quota. Leverage shared drives (https://developers.google.com/workspace/drive/api/guides/about-shareddrives), or use OAuth delegation (http://support.google.com/a/answer/7281227) instead.', 'domain': 'usageLimits', 'reason': 'storageQuotaExceeded'}]">
  * NEW BEST 0.3937


E2 val: 100%|██████████| 359/359 [05:04<00:00,  1.18it/s]


E2/50 | loss 0.6493/0.5616 | mIoU 0.4687 | 2225s
  [DRIVE FAIL] <HttpError 403 when requesting None returned "Service Accounts do not have storage quota. Leverage shared drives (https://developers.google.com/workspace/drive/api/guides/about-shareddrives), or use OAuth delegation (http://support.google.com/a/answer/7281227) instead.". Details: "[{'message': 'Service Accounts do not have storage quota. Leverage shared drives (https://developers.google.com/workspace/drive/api/guides/about-shareddrives), or use OAuth delegation (http://support.google.com/a/answer/7281227) instead.', 'domain': 'usageLimits', 'reason': 'storageQuotaExceeded'}]">
  * NEW BEST 0.4687


E3 val: 100%|██████████| 359/359 [04:41<00:00,  1.28it/s]


...

E9/50 | loss 0.4903/0.4691 | mIoU 0.5247 | 2124s


E10 val: 100%|██████████| 359/359 [04:43<00:00,  1.27it/s]


E10/50 | loss 0.4800/0.4506 | mIoU 0.5337 | 2123s


E11 train:   6%|▋         | 142/2188 [01:59<28:47,  1.18it/s]


KeyboardInterrupt: 

In [ ]:
# ============================================================
# DOWNLOAD CHECKPOINTS
# ============================================================
import shutil
zip_path = '/kaggle/working/checkpoints_backup'
shutil.make_archive(zip_path, 'zip', str(CKPT_DIR))
print(f'Zip created: {zip_path}.zip ({Path(zip_path + ".zip").stat().st_size/1024**2:.1f}MB)')
print('Go to Output tab to download')

Zip created: /kaggle/working/checkpoints_backup.zip (804.3MB)
Go to Output tab to download


In [ ]:
# === AUTO-UPLOAD CHECKPOINT AS KAGGLE DATASET ===
import json, subprocess

KAGGLE_USER = "your_kaggle_username"  # đổi thành username của bạn
DS_SLUG = f"{KAGGLE_USER}/forest-seg-checkpoints"

# Tạo metadata
meta = {
    "title": "Forest Seg Checkpoints",
    "id": DS_SLUG,
    "licenses": [{"name": "CC0-1.0"}]
}
meta_path = CKPT_DIR / "dataset-metadata.json"
with open(meta_path, "w") as f:
    json.dump(meta, f)

# Upload (lần đầu dùng create, lần sau dùng version)
try:
    subprocess.run(["kaggle", "datasets", "version", "-p", str(CKPT_DIR),
                    "-m", f"best_miou_{best_miou:.4f}"], check=True)
except:
    subprocess.run(["kaggle", "datasets", "create", "-p", str(CKPT_DIR)], check=True)

print(f"[OK] Uploaded to kaggle.com/datasets/{DS_SLUG}")

Starting upload for file epoch_006_miou_0.5176.pth


100%|██████████| 291M/291M [00:05<00:00, 57.6MB/s] 
  0%|          | 0.00/291M [00:00<?, ?B/s]

Upload successful: epoch_006_miou_0.5176.pth (291MB)
Starting upload for file epoch_008_miou_0.5376.pth


100%|██████████| 291M/291M [00:03<00:00, 81.6MB/s] 
  6%|▌         | 16.6M/291M [00:00<00:01, 171MB/s]

Upload successful: epoch_008_miou_0.5376.pth (291MB)
Starting upload for file epoch_007_miou_0.5295.pth


...

100%|██████████| 291M/291M [00:03<00:00, 94.8MB/s]
  3%|▎         | 8.66M/291M [00:00<00:03, 79.1MB/s]

Upload successful: epoch_008_miou_0.5376.pth (291MB)
Starting upload for file epoch_007_miou_0.5295.pth
Error while trying to load upload info: KaggleObject.from_dict() got an unexpected keyword argument 'token'


 99%|█████████▉| 288M/291M [00:03<00:00, 92.1MB/s] 

Upload successful: epoch_007_miou_0.5295.pth (291MB)
Dataset creation error: Invalid Owner Id
[OK] Uploaded to kaggle.com/datasets/your_kaggle_username/forest-seg-checkpoints


100%|██████████| 291M/291M [00:03<00:00, 84.3MB/s]
